# Does acting on a detected failure help? A pre-registered experiment

Reviewer 3's seventh comment asks for it, and the submitted version answered that
detection is a precondition for intervention and that we would rather not improvise an
experiment whose design choices would carry the result. The second half of that answer
is a real risk and this file is how it is managed: **every choice is fixed here, in the
repository, before the experiment is run**. Nothing below is decided after seeing a
number.

Read this file as the protocol. It is committed to the artifact before any run, so the
commit date bounds when the choices were made, and the deviation rule at the end binds
anything that changes afterwards.

## What is fixed

**Task set.** `benchmark.generate(per_template=25, seed=42)`, the corrected generator,
200 tasks. These are the same kinds of task as the study's corpus and not the same
sequence; the original draw order is not recoverable, which `benchmark.py` states. Both
arms receive the identical list, so the comparison does not depend on it.

**Agent and decoding.** Qwen 2.5 1.5B-Instruct, greedy, `max_new_tokens=150`, six-action
budget, the tool surface and prompt of `code/agent.py`. The prompt is passed through the
tokenizer's chat template as one user message, which is a choice: the original run's
prompt is not reproducible character for character. Both arms share it.

**Arm A, baseline.** `run_agent` from the released `agent.py`, unchanged.

**Arm B, intervention.** The same loop, plus the two free signals of Section IV.B
evaluated as the trajectory grows. After each step, `tier1` runs on the trajectory so
far; on a `finish`, the contradiction half of `tier2` runs on the final answer. On a
flag, the step is kept in the trace, a fixed corrective note is appended to the context,
and the agent takes that step again; a flagged `finish` does not terminate the
trajectory. At most **two** interventions per trajectory, retries do not consume the
six-action budget, and a hard ceiling of ten steps bounds the cost. The duplicate test
is not used as a trigger, because Section VI.C reports that it detects nothing on this
corpus at the published cutoff.

**The two notes, verbatim and in the agent's language.** They are deliberately generic:
they name the kind of defect and ask for the step again, and they never supply the
answer.

**Outcome.** End-to-end success, decided by `code/experiments/success_checker.py`, frozen
and validated before this experiment existed: it recovers 72 of 72 human successes on the
study corpus and accepts nine more that end correctly after an earlier fault. The
primary measure is the paired difference in success rate, arm B minus arm A, over the 200
tasks. The test is the exact binomial on the discordant pairs. The interval is a cluster
bootstrap over the eight templates, which is few clusters and will be reported as such.
Secondary measures, also fixed now: interventions fired, extra steps consumed, the
per-template difference, and the count of trajectories where an intervention fired and
the answer stayed wrong.

## What is predicted, so that it cannot be rewritten afterwards

The signals flag 53.1 per cent of failures on the study corpus and almost none of the
reasoning and retrieval errors, which are 69 of its 128 failures. An intervention can
only fire on what the signals see, so the effect available to this experiment is bounded
by the structural failures and is expected to be small. A null result is a likely
outcome, and a negative one is possible, since a retry consumes budget and can replace a
wrong answer with an empty one. Whatever comes out is what gets reported.

## Deviation rule

If anything in this protocol has to change once the run starts, the change is written
into this file with its reason and the date, and the result is reported as a deviation
rather than as the plan.

**Requires:** the refreshed `neucom-d-26-17269-r1` tag, which carries the corrected
generator and the frozen checker. GPU T4, internet on.

## Deviations

**Deviation 1, 19 October 2026, before any result existed.** The first launch stopped on
the first cell with `NameError: name '__file__' is not defined`. The repository modules
are fetched and executed from a string, so they have no `__file__`, and
`success_checker.py` uses it at import time to locate `data/`. The loader now supplies
the name; the checker itself is unchanged and still passes its own self-check in the
cell below, 72 of 72. Nothing in the protocol, the arms, the notes, the caps or the
analysis changed, and no number had been produced when this was fixed.


In [ ]:
import ast, json, re, sys, time, types, urllib.request
import torch
from collections import Counter

TAG = ("https://raw.githubusercontent.com/achenna-source/"
       "icv-agent-observability/neucom-d-26-17269-r1/")
fetch = lambda p: urllib.request.urlopen(TAG + p, timeout=90).read().decode("utf-8")

def load_module(path, name):
    mod = types.ModuleType(name)
    mod.__dict__["__name__"] = name
    # Deviation 1 : les modules du depot sont executes depuis une chaine, donc __file__
    # n'existe pas et success_checker.py s'en sert au chargement pour situer data/.
    # Le nom est fourni ici ; le module lui-meme n'est pas touche.
    mod.__dict__["__file__"] = "/kaggle/working/" + path
    exec(compile(fetch(path), path, "exec"), mod.__dict__)
    sys.modules[name] = mod
    return mod

agent = load_module("code/agent.py", "agent")
signals = load_module("code/signals.py", "signals")
benchmark = load_module("code/benchmark.py", "benchmark")
checker = load_module("code/experiments/success_checker.py", "success_checker")

_kb = json.loads(fetch("data/knowledge_base.json"))
KB = _kb["facts"]
POP = {k.split(" ", 1)[1]: int(v) for k, v in KB.items() if k.startswith("population ")}
CAP = {k.split(" ", 1)[1]: v for k, v in KB.items() if k.startswith("capitale ")}
KNOWN = {int(v) for v in KB.values() if str(v).isdigit()}
STORE = (POP, CAP, KNOWN)

rows = [json.loads(l) for l in fetch("data/trajectories_200.jsonl").splitlines()
        if l.strip()]
print("modules loaded from the tag: agent, signals, benchmark, success_checker")
print("gpus: %d" % torch.cuda.device_count())


## The two frozen checks before anything runs

The generator must agree with the corpus it documents, and the checker must still
recover every human success. If either assertion fails, the protocol is void and the run
does not proceed.


In [ ]:
import io, os, tempfile
_tmp = os.path.join(tempfile.gettempdir(), "trajectories_200.jsonl")
io.open(_tmp, "w", encoding="utf-8").write(fetch("data/trajectories_200.jsonl"))
print("generator self-check : %s" % benchmark.self_check(corpus_path=_tmp))

_kbtmp = os.path.join(tempfile.gettempdir(), "knowledge_base.json")
io.open(_kbtmp, "w", encoding="utf-8").write(fetch("data/knowledge_base.json"))
checker.DATA = tempfile.gettempdir()
print("checker self-check   : %s" % checker.self_check(corpus_path=_tmp))
assert checker.self_check(corpus_path=_tmp)["recovered"] == 72, "checker drifted"
print()
print("both frozen components verify; the protocol stands")


## Arm B, written out

The intervention is the only difference from `agent.run_agent`, and it is visible here
in full rather than described.


In [ ]:
NOTE_SCHEMA = ("Observation: Erreur de forme detectee a l'etape precedente. Verifie que "
               "le nom d'outil est l'un de knowledge_base, calculator, finish, et que "
               "l'argument est non vide. Reprends cette etape.")
NOTE_CONTRA = ("Observation: Verification: la reponse finale contient une valeur qui ne "
               "provient pas des observations obtenues. Reprends en t'appuyant "
               "uniquement sur les valeurs recuperees, ou indique que l'information "
               "est absente.")
MAX_INTERVENTIONS = 2
MAX_STEPS_HARD = 10
BUDGET = 6

def run_with_intervention(task, llm_generate, kb):
    tools = {"calculator": agent.calculator,
             "knowledge_base": agent.make_knowledge_base(kb)}
    trajectory, fired, actions, total = [], [], 0, 0
    context = agent.SYSTEM_PROMPT + "\n\nTache : %s\n\n" % task
    while actions < BUDGET and total < MAX_STEPS_HARD:
        total += 1
        parsed = agent.parse_action(llm_generate(context))
        action, inp = parsed["action"], parsed["input"]
        if action == "finish":
            step = {"thought": parsed["thought"], "action": "finish",
                    "input": inp, "observation": None}
            candidate = trajectory + [step]
            _, _, contra = signals.tier2(candidate, encoder=None)
            if contra and len(fired) < MAX_INTERVENTIONS:
                trajectory.append(step)
                fired.append(("contradiction", total))
                context += ('Thought: %s\nAction: finish\nAction Input: "%s"\n%s\n'
                            % (parsed["thought"], inp, NOTE_CONTRA))
                continue
            trajectory.append(step)
            break
        obs = (tools[action](inp) if action in tools and inp
               else "Erreur: outil inconnu '%s'" % action)
        step = {"thought": parsed["thought"], "action": action,
                "input": inp, "observation": obs}
        trajectory.append(step)
        flagged, _ = signals.tier1(trajectory)
        if flagged and len(fired) < MAX_INTERVENTIONS:
            fired.append(("schema", total))
            context += ('Thought: %s\nAction: %s\nAction Input: "%s"\n%s\n'
                        % (parsed["thought"], action, inp, NOTE_SCHEMA))
            continue
        actions += 1
        context += ('Thought: %s\nAction: %s\nAction Input: "%s"\nObservation: %s\n'
                    % (parsed["thought"], action, inp, obs))
    return {"task": task, "trajectory": trajectory,
            "interventions": fired, "steps_generated": total}

print("arm B defined: %d interventions at most, hard ceiling %d steps"
      % (MAX_INTERVENTIONS, MAX_STEPS_HARD))


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16,
                                             device_map="auto")
model.eval()

def llm_generate(prompt):
    msgs = [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=150, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

TASKS = benchmark.generate(per_template=25, seed=42)
print("%s loaded | tasks %d | templates %s"
      % (MODEL, len(TASKS), dict(Counter(t["template"] for t in TASKS))))


In [ ]:
def run_arm(fn, tag):
    out, t0 = [], time.time()
    for i, t in enumerate(TASKS):
        r = fn(t["task"])
        r["template"] = t["template"]
        out.append(r)
        if (i + 1) % 25 == 0:
            print("   [%s] %3d/200  %.0f s" % (tag, i + 1, time.time() - t0))
    return out

A = run_arm(lambda task: agent.run_agent(task, llm_generate, KB), "A baseline")
json.dump(A, open("arm_a.json", "w"), ensure_ascii=False)
print("arm A done")


In [ ]:
B = run_arm(lambda task: run_with_intervention(task, llm_generate, KB), "B intervene")
json.dump(B, open("arm_b.json", "w"), ensure_ascii=False)
print("arm B done")


## The pre-specified analysis

Nothing here chooses a statistic after seeing the data: the primary measure, the test and
the interval were fixed above.


In [ ]:
def score(arm):
    return [checker.succeeded(r["task"], r["trajectory"], STORE)[0] for r in arm]

sa, sb = score(A), score(B)
both = sum(1 for a, b in zip(sa, sb) if a and b)
only_a = sum(1 for a, b in zip(sa, sb) if a and not b)
only_b = sum(1 for a, b in zip(sa, sb) if b and not a)
neither = sum(1 for a, b in zip(sa, sb) if not a and not b)

print("arm A success %d/200 = %.1f%%" % (sum(sa), 100.0 * sum(sa) / 200))
print("arm B success %d/200 = %.1f%%" % (sum(sb), 100.0 * sum(sb) / 200))
print("paired difference %+.1f points" % (100.0 * (sum(sb) - sum(sa)) / 200))
print()
print("both %d | only A %d | only B %d | neither %d" % (both, only_a, only_b, neither))

from math import comb
n, k = only_a + only_b, only_b
if n:
    p = sum(comb(n, i) for i in range(min(k, n - k) + 1)) / 2 ** n * 2
    print("exact binomial on the %d discordant pairs: p = %.4f" % (n, min(1.0, p)))
else:
    print("no discordant pair: the two arms agree on every task")


In [ ]:
import random
rng = random.Random(42)
tpls = sorted({t["template"] for t in TASKS})
idx = {t: [i for i, x in enumerate(TASKS) if x["template"] == t] for t in tpls}
draws = []
for _ in range(5000):
    pick = [rng.choice(tpls) for _ in tpls]
    sel = [i for t in pick for i in idx[t]]
    draws.append(sum(sb[i] for i in sel) / len(sel) - sum(sa[i] for i in sel) / len(sel))
draws.sort()
lo, hi = draws[int(0.025 * len(draws))], draws[int(0.975 * len(draws))]
print("cluster bootstrap over the 8 templates: %+.3f [%+.3f, %+.3f]"
      % (sum(sb) / 200 - sum(sa) / 200, lo, hi))
print("eight clusters is few, and the interval is wide for that reason")
print()
print("%-5s %8s %8s %8s" % ("tmpl", "A", "B", "diff"))
for t in tpls:
    ia = [i for i in idx[t]]
    a = sum(sa[i] for i in ia)
    b = sum(sb[i] for i in ia)
    print("%-5s %8d %8d %+8d" % (t, a, b, b - a))


In [ ]:
fired = [len(r["interventions"]) for r in B]
kinds = Counter(k for r in B for k, _ in r["interventions"])
extra = sum(r["steps_generated"] for r in B) - sum(len(r["trajectory"]) for r in A)
still_wrong = sum(1 for r, ok in zip(B, sb) if r["interventions"] and not ok)
print("trajectories with at least one intervention : %d" % sum(1 for f in fired if f))
print("interventions by kind                       : %s" % dict(kinds))
print("generated steps, arm B minus arm A          : %+d" % extra)
print("intervened and still wrong                  : %d" % still_wrong)
json.dump({"A": sum(sa), "B": sum(sb), "only_a": only_a, "only_b": only_b,
           "ci": [round(lo, 3), round(hi, 3)], "fired": fired,
           "kinds": dict(kinds), "still_wrong": still_wrong,
           "score_a": sa, "score_b": sb},
          open("intervention_result.json", "w"))
print("written: intervention_result.json")
